# 旅行客户支持与交接编排

本笔记本展示了使用 Microsoft Agent Framework 的**交接编排**。我们将构建一个旅行客户支持系统，代理可以根据客户需求将控制权转交给专家。

## 您将学习：
1. **交接编排**：基于上下文和专业知识的动态代理路由
2. **HandoffBuilder**：用于构建交接工作流的高级 API
3. **专家路由**：代理可以动态地交接给其他代理
4. **多轮对话**：在交接过程中无缝保留上下文
5. **客户支持流程**：代理交接的实际应用


## 前提条件：
- 已安装 Microsoft Agent Framework
- 已配置 GitHub 令牌或 OpenAI API 密钥
- 了解基本代理概念


In [17]:
import asyncio
import json
import os
from collections.abc import AsyncIterable
from typing import Any

from agent_framework import Agent, Message
from agent_framework._workflows._events import WorkflowEvent, WorkflowRunState
from agent_framework.openai import OpenAIChatCompletionClient
from agent_framework.orchestrations import HandoffAgentUserRequest, HandoffBuilder
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel


## 第一步：定义 Pydantic 模型以生成结构化输出

这些模型定义了每个专用代理将返回的模式。这可以确保所有代理的响应一致且可解析。


In [18]:
# 定义 `FlightBookingResult` 类，用来封装一组相关的数据或行为。
class FlightBookingResult(BaseModel):
    """Flight booking confirmation from the booking agent."""

    destination: str
    departure_date: str
    return_date: str
    booking_reference: str
    passenger_name: str
    flight_details: str
    total_cost: str
    status: str


# 定义 `DisputeResult` 类，用来封装一组相关的数据或行为。
class DisputeResult(BaseModel):
    """Dispute resolution result from the disputes agent."""

    dispute_type: str
    original_booking: str
    refund_amount: str
    refund_method: str
    processing_time: str
    reference_number: str
    status: str


# 定义 `TripCheckResult` 类，用来封装一组相关的数据或行为。
class TripCheckResult(BaseModel):
    """Trip confirmation result from the trip check agent."""

    trip_reference: str
    destination: str
    travel_dates: str
    confirmation_status: str
    special_notes: str
    contact_info: str


## 第2步：加载环境变量


In [19]:
# Load environment variables
# 从 `.env` 文件加载环境变量配置。
load_dotenv()

# 创建兼容 OpenAI 接口的聊天客户端，用来连接模型服务。
chat_client = OpenAIChatCompletionClient(
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    api_key=os.environ.get("DASHSCOPE_API_KEY"),
    model="qwen-max",
)

print("Chat client configured with DashScope qwen-max")


Chat client configured with DashScope qwen-max


## 第三步：创建四个专业的旅行支持代理

每个代理都有特定的专业领域，并可以根据客户需求转交给适当的专家。


In [20]:
# Agent 1: Customer Support Agent (Main triage agent)
customer_support_agent = Agent(
    client=chat_client,
    instructions=(
        "You are a friendly customer support agent for a travel company. "
        "Assess customer requests and route them to the appropriate specialist: "
        "- For flight bookings or reservations: call handoff_to_booking_agent "
        "- For refunds, disputes, or billing issues: call handoff_to_disputes_agent "
        "- For trip confirmations or travel plan checks: call handoff_to_trip_check_agent "
        "Be welcoming and ensure customers feel heard before routing them."
    ),
    name="customer_support_agent",
    require_per_service_call_history_persistence=True,
)

# Agent 2: Booking Agent (Flight booking specialist)
booking_agent = Agent(
    client=chat_client,
    instructions=(
        "You are a flight booking specialist. Handle all flight reservations and bookings. "
        "When a customer wants to book a flight, collect their destination, travel dates, "
        "and confirm the booking. Always provide a booking reference number. "
        "Return structured JSON with booking details. "
        "The flight is always confirmed as booked regardless of destination. "
        "You MUST return valid JSON only."
    ),
    name="booking_agent",
    default_options={"response_format": FlightBookingResult},
    require_per_service_call_history_persistence=True,
)

# Agent 3: Disputes Agent (Refund and billing specialist)
disputes_agent = Agent(
    client=chat_client,
    instructions=(
        "You are a disputes and refunds specialist. Handle customer complaints, "
        "refund requests, and billing disputes. Always approve refunds and provide "
        "a reference number. Process refunds back to the original payment method. "
        "Return structured JSON with refund details. "
        "All refund requests are approved and processed immediately. "
        "You MUST return valid JSON only."
    ),
    name="disputes_agent",
    default_options={"response_format": DisputeResult},
    require_per_service_call_history_persistence=True,
)

# Agent 4: Trip Check Agent (Travel confirmation specialist)
trip_check_agent = Agent(
    client=chat_client,
    instructions=(
        "You are a travel confirmation specialist. Verify and confirm customer "
        "travel plans, check itineraries, and provide travel status updates. "
        "Always confirm that travel plans are in order and provide reassurance. "
        "Return structured JSON with confirmation details. "
        "All travel plans are confirmed as valid and ready. "
        "You MUST return valid JSON only."
    ),
    name="trip_check_agent",
    default_options={"response_format": TripCheckResult},
    require_per_service_call_history_persistence=True,
)


## 第四步：构建交接工作流程

HandoffBuilder 创建了一个工作流程，使客户支持代理可以根据客户需求动态地交接给专家。


In [21]:
workflow = (
    # 创建 handoff 工作流构建器，让 Agent 可以相互转交任务。
    HandoffBuilder(
        name="travel_support_handoff",
        participants=[customer_support_agent, booking_agent, disputes_agent, trip_check_agent],
    )
    .with_start_agent(customer_support_agent)
    .add_handoff(customer_support_agent, [booking_agent, disputes_agent, trip_check_agent])
    .with_termination_condition(
        lambda conv: sum(1 for msg in conv if msg.role == "user") > 3
    )
    # 根据前面配置生成最终可运行的工作流对象。
    .build()
)

display(HTML("""
<div style='padding: 20px; background: linear-gradient(135deg, #ff7043 0%, #ff5722 100%); color: white; border-radius: 8px; margin: 10px 0;'>
    <h3 style='margin: 0 0 15px 0;'>Handoff Workflow Built Successfully!</h3>
    <p style='margin: 0; line-height: 1.6;'>
        <strong>Handoff Flow:</strong><br>
        • User Request → <strong>Customer Support Agent</strong> (triage)<br>
        • Support Agent → <strong>Specialist Agent</strong> (dynamic handoff)<br>
        • Specialist → <strong>Resolution</strong> (expert handling)<br>
        • System → <strong>User Response</strong> (final result)
    </p>
</div>
"""))


No handoff configuration found for agent 'booking_agent'. This agent will not be able to hand off to any other agents and your workflow may get stuck.
No handoff configuration found for agent 'disputes_agent'. This agent will not be able to hand off to any other agents and your workflow may get stuck.
No handoff configuration found for agent 'trip_check_agent'. This agent will not be able to hand off to any other agents and your workflow may get stuck.


## 第五步：事件处理辅助函数

这些函数帮助我们处理工作流事件，并在交接过程中处理用户输入请求。


In [22]:
# 定义异步函数 `drain_events`，用于处理需要 `await` 的流程。
async def drain_events(stream: AsyncIterable[WorkflowEvent]) -> list[WorkflowEvent]:
    """Collect all events from an async stream into a list."""
    # 返回当前函数的结果给调用方。
    return [event async for event in stream]


# 定义函数 `extract_messages`，把一段可复用逻辑封装起来。
def extract_messages(data: Any) -> list[Message]:
    """Normalize workflow event data to a list of messages."""
    # 根据当前条件决定后续走哪条逻辑分支。
    if hasattr(data, "messages"):
        # 返回当前函数的结果给调用方。
        return list(data.messages)
    # 根据当前条件决定后续走哪条逻辑分支。
    if isinstance(data, list):
        # 返回当前函数的结果给调用方。
        return [msg for msg in data if isinstance(msg, Message)]
    # 返回当前函数的结果给调用方。
    return []


# 定义函数 `handle_workflow_events`，把一段可复用逻辑封装起来。
def handle_workflow_events(events: list[WorkflowEvent]) -> list[WorkflowEvent]:
    """Process workflow events and extract pending user input requests."""
    requests: list[WorkflowEvent] = []

    # 遍历这一组数据，逐条处理。
    for event in events:
        # 根据当前条件决定后续走哪条逻辑分支。
        if event.type == "status" and event.state in {
            WorkflowRunState.IDLE,
            WorkflowRunState.IDLE_WITH_PENDING_REQUESTS,
        }:
            print(f"[Workflow Status] {event.state.name}")

        # 如果前面的条件不成立，再判断这个分支条件。
        elif event.type == "output":
            conversation = extract_messages(event.data)
            # 根据当前条件决定后续走哪条逻辑分支。
            if conversation:
                print("\n=== Final Conversation ===")
                # 遍历这一组数据，逐条处理。
                for message in conversation:
                    # 根据当前条件决定后续走哪条逻辑分支。
                    if not message.text.strip():
                        continue
                    speaker = message.author_name or message.role
                    print(f"- {speaker}: {message.text}")
                print("==========================")

        # 如果前面的条件不成立，再判断这个分支条件。
        elif event.type == "request_info" and isinstance(event.data, HandoffAgentUserRequest):
            print_handoff_request(event.data)
            # 向列表末尾追加一个新元素。
            requests.append(event)

    # 返回当前函数的结果给调用方。
    return requests


# 定义函数 `print_handoff_request`，把一段可复用逻辑封装起来。
def print_handoff_request(request: HandoffAgentUserRequest) -> None:
    """Display a user input request with conversation context."""
    print("\n=== User Input Requested ===")
    # 去掉字符串首尾的空白字符，便于后续判断。
    messages_with_text = [msg for msg in request.agent_response.messages if msg.text.strip()]
    print(f"Last {len(messages_with_text)} messages in conversation:")
    # 遍历这一组数据，逐条处理。
    for message in messages_with_text[-3:]:
        speaker = message.author_name or message.role
        text = message.text[:100] + "..." if len(message.text) > 100 else message.text
        print(f"  {speaker}: {text}")
    print("============================")


print("Helper functions defined for event processing")


Helper functions defined for event processing


## 第六步：测试用例1 - 机票预订请求

让我们用一个机票预订请求来测试我们的交接流程。客服人员应该将请求交接给预订代理。


In [23]:
# 定义异步函数 `test_booking_handoff`，用于处理需要 `await` 的流程。
async def test_booking_handoff():
    """Test handoff workflow for flight booking requests."""

    display(HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>Test Case 1: Flight Booking Request</h3>
        <p style='margin: 0;'><strong>Expected Flow:</strong> Customer Support → Booking Agent</p>
    </div>
    """))

    # Start the workflow
    print("[User]: I want to book a flight to Paris for next month")
    events = await drain_events(
        # 以流式方式启动工作流，边执行边接收事件。
        workflow.run("I want to book a flight to Paris for next month", stream=True)
    )
    pending_requests = handle_workflow_events(events)

    # Handle any additional user input requests
    scripted_responses = [
        "I'd like to travel from New York to Paris on December 15th and return on December 22nd.",
        "Yes, please confirm the booking under the name John Smith."
    ]

    response_index = 0
    # 当条件满足时持续循环执行下面的代码。
    # 循环执行，直到条件不再满足。
    # 循环执行，直到条件不再满足。
    # 循环执行，直到条件不再满足。
    while pending_requests and response_index < len(scripted_responses):
        user_response = scripted_responses[response_index]
        print(f"\n[User]: {user_response}")

        responses = {req.request_id: HandoffAgentUserRequest.create_response(user_response) for req in pending_requests}
        # 把人工输入或补充响应继续送回工作流中。
        events = await drain_events(workflow.run(stream=True, responses=responses))
        pending_requests = handle_workflow_events(events)

        response_index += 1

    # Extract and display the final booking result
    if events:
        # 遍历这一组数据，逐条处理。
        for event in events:
            # 根据当前条件决定后续走哪条逻辑分支。
            if event.type == "output":
                conversation = extract_messages(event.data)
                # 遍历这一组数据，逐条处理。
                for message in conversation:
                    # 根据当前条件决定后续走哪条逻辑分支。
                    if message.author_name == "booking_agent" and message.text.strip():
                        # 尝试执行可能出错的代码，便于后面做异常处理。
                        try:
                            # 把 JSON 字符串校验并解析成 Pydantic 结构化对象。
                            booking_data = FlightBookingResult.model_validate_json(
                                message.text)
                            display_booking_result(booking_data)
                        # 捕获前面代码抛出的异常，避免程序直接中断。
                        except Exception as e:
                            print(f"Could not parse booking result: {e}")


# 定义函数 `display_booking_result`，把一段可复用逻辑封装起来。
def display_booking_result(booking: FlightBookingResult):
    """Display flight booking result in a formatted section."""

    display(HTML(f"""
    <div style='padding: 20px; background: #e8f5e9; border-radius: 8px; margin: 15px 0; border-left: 4px solid #4caf50;'>
        <h3 style='margin: 0 0 15px 0; color: #2e7d32;'>✈️ Flight Booking Confirmed</h3>
        <div style='display: grid; grid-template-columns: 1fr 1fr; gap: 15px; margin-bottom: 15px;'>
            <div>
                <strong style='color: #333;'>Booking Reference:</strong> {booking.booking_reference}<br>
                <strong style='color: #333;'>Passenger:</strong> {booking.passenger_name}<br>
                <strong style='color: #333;'>Status:</strong> <span style='color: #4caf50; font-weight: bold;'>{booking.status}</span>
            </div>
            <div>
                <strong style='color: #333;'>Destination:</strong> {booking.destination}<br>
                <strong style='color: #333;'>Total Cost:</strong> {booking.total_cost}<br>
                <strong style='color: #333;'>Departure:</strong> {booking.departure_date}
            </div>
        </div>
        <div style='margin-bottom: 10px;'>
            <strong style='color: #333;'>Flight Details:</strong> {booking.flight_details}
        </div>
        <div style='background: rgba(76,175,80,0.1); padding: 10px; border-radius: 4px; margin-top: 10px;'>
            <strong style='color: #2e7d32;'>✅ Success:</strong> Flight booking completed through handoff to booking specialist
        </div>
    </div>
    """))


# Run the booking test
# 等待异步操作完成，再继续执行后续代码。
await test_booking_handoff()


[User]: I want to book a flight to Paris for next month

=== User Input Requested ===
Last 1 messages in conversation:
  booking_agent: {
  "destination": "Paris",
  "travel_dates": "Next month",
  "booking_status": "Confirmed",
  "book...
[Workflow Status] IDLE_WITH_PENDING_REQUESTS

[User]: I'd like to travel from New York to Paris on December 15th and return on December 22nd.

=== User Input Requested ===
Last 1 messages in conversation:
  booking_agent: {
  "departure_city": "New York",
  "destination": "Paris",
  "departure_date": "December 15th",
  "...
[Workflow Status] IDLE_WITH_PENDING_REQUESTS

[User]: Yes, please confirm the booking under the name John Smith.

=== User Input Requested ===
Last 1 messages in conversation:
  booking_agent: {
  "passenger_name": "John Smith",
  "departure_city": "New York",
  "destination": "Paris",
  "dep...
[Workflow Status] IDLE_WITH_PENDING_REQUESTS


## 第7步：测试用例2 - 争议/退款请求

让我们用退款请求来测试我们的交接流程。客户支持代理应将请求交接给争议处理代理。


In [24]:
# 定义异步函数 `test_dispute_handoff`，用于处理需要 `await` 的流程。
async def test_dispute_handoff():
    """Test handoff workflow for dispute/refund requests."""

    display(HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>Test Case 2: Refund Request</h3>
        <p style='margin: 0;'><strong>Expected Flow:</strong> Customer Support → Disputes Agent</p>
    </div>
    """))

    # Start the workflow
    print("[User]: I need to cancel my flight and get a refund")
    events = await drain_events(
        # 以流式方式启动工作流，边执行边接收事件。
        workflow.run("I need to cancel my flight and get a refund", stream=True)
    )
    pending_requests = handle_workflow_events(events)

    # Handle any additional user input requests
    scripted_responses = [
        "My booking reference is FL12345. I can't travel due to a family emergency.",
        "Yes, please process the full refund back to my credit card."
    ]

    response_index = 0
    # 当条件满足时持续循环执行下面的代码。
    # 循环执行，直到条件不再满足。
    # 循环执行，直到条件不再满足。
    # 循环执行，直到条件不再满足。
    while pending_requests and response_index < len(scripted_responses):
        user_response = scripted_responses[response_index]
        print(f"\n[User]: {user_response}")

        responses = {req.request_id: HandoffAgentUserRequest.create_response(user_response) for req in pending_requests}
        # 把人工输入或补充响应继续送回工作流中。
        events = await drain_events(workflow.run(stream=True, responses=responses))
        pending_requests = handle_workflow_events(events)

        response_index += 1

    # Extract and display the final dispute result
    if events:
        # 遍历这一组数据，逐条处理。
        for event in events:
            # 根据当前条件决定后续走哪条逻辑分支。
            if event.type == "output":
                conversation = extract_messages(event.data)
                # 遍历这一组数据，逐条处理。
                for message in conversation:
                    # 根据当前条件决定后续走哪条逻辑分支。
                    if message.author_name == "disputes_agent" and message.text.strip():
                        # 尝试执行可能出错的代码，便于后面做异常处理。
                        try:
                            # 把 JSON 字符串校验并解析成 Pydantic 结构化对象。
                            dispute_data = DisputeResult.model_validate_json(
                                message.text)
                            display_dispute_result(dispute_data)
                        # 捕获前面代码抛出的异常，避免程序直接中断。
                        except Exception as e:
                            print(f"Could not parse dispute result: {e}")


# 定义函数 `display_dispute_result`，把一段可复用逻辑封装起来。
def display_dispute_result(dispute: DisputeResult):
    """Display dispute resolution result in a formatted section."""

    display(HTML(f"""
    <div style='padding: 20px; background: #fff3e0; border-radius: 8px; margin: 15px 0; border-left: 4px solid #ff9800;'>
        <h3 style='margin: 0 0 15px 0; color: #f57c00;'>💰 Refund Processed</h3>
        <div style='display: grid; grid-template-columns: 1fr 1fr; gap: 15px; margin-bottom: 15px;'>
            <div>
                <strong style='color: #333;'>Reference Number:</strong> {dispute.reference_number}<br>
                <strong style='color: #333;'>Dispute Type:</strong> {dispute.dispute_type}<br>
                <strong style='color: #333;'>Status:</strong> <span style='color: #ff9800; font-weight: bold;'>{dispute.status}</span>
            </div>
            <div>
                <strong style='color: #333;'>Refund Amount:</strong> {dispute.refund_amount}<br>
                <strong style='color: #333;'>Refund Method:</strong> {dispute.refund_method}<br>
                <strong style='color: #333;'>Processing Time:</strong> {dispute.processing_time}
            </div>
        </div>
        <div style='margin-bottom: 10px;'>
            <strong style='color: #333;'>Original Booking:</strong> {dispute.original_booking}
        </div>
        <div style='background: rgba(255,152,0,0.1); padding: 10px; border-radius: 4px; margin-top: 10px;'>
            <strong style='color: #f57c00;'>✅ Success:</strong> Refund processed through handoff to disputes specialist
        </div>
    </div>
    """))

    # Run the dispute test
# 等待异步操作完成，再继续执行后续代码。
await test_dispute_handoff()


[User]: I need to cancel my flight and get a refund
[Workflow Status] IDLE


## 第8步：测试用例3 - 行程确认请求

让我们通过行程确认请求来测试我们的交接工作流程。客户支持代理应将请求交接给行程检查代理。


In [27]:
# 定义异步函数 `test_trip_check_handoff`，用于处理需要 `await` 的流程。
async def test_trip_check_handoff():
    """Test handoff workflow for trip confirmation requests."""

    display(HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>Test Case 3: Trip Confirmation</h3>
        <p style='margin: 0;'><strong>Expected Flow:</strong> Customer Support → Trip Check Agent</p>
    </div>
    """))

    # Start the workflow
    print("[User]: Can you confirm my travel plans are all set?")
    events = await drain_events(
        # 以流式方式启动工作流，边执行边接收事件。
        workflow.run("Can you confirm my travel plans are all set?", stream=True)
    )
    pending_requests = handle_workflow_events(events)

    # Handle any additional user input requests
    scripted_responses = [
        "I'm traveling to London next week. My confirmation number is TR98765.",
        "Perfect, thank you for checking everything is ready!"
    ]

    response_index = 0
    # 当条件满足时持续循环执行下面的代码。
    # 循环执行，直到条件不再满足。
    # 循环执行，直到条件不再满足。
    # 循环执行，直到条件不再满足。
    while pending_requests and response_index < len(scripted_responses):
        user_response = scripted_responses[response_index]
        print(f"\n[User]: {user_response}")
        
        responses = {req.request_id: HandoffAgentUserRequest.create_response(user_response) for req in pending_requests}
        # 把人工输入或补充响应继续送回工作流中。
        events = await drain_events(workflow.run(stream=True, responses=responses))
        pending_requests = handle_workflow_events(events)
        
        response_index += 1
    # Extract and display the final trip check result
    if events:
        # 遍历这一组数据，逐条处理。
        for event in events:
            # 根据当前条件决定后续走哪条逻辑分支。
            if event.type == "output":
                conversation = extract_messages(event.data)
                # 遍历这一组数据，逐条处理。
                for message in conversation:
                    # 根据当前条件决定后续走哪条逻辑分支。
                    if message.author_name == "trip_check_agent" and message.text.strip():
                        # 尝试执行可能出错的代码，便于后面做异常处理。
                        try:
                            # 把 JSON 字符串校验并解析成 Pydantic 结构化对象。
                            trip_data = TripCheckResult.model_validate_json(
                                message.text)
                            display_trip_check_result(trip_data)
                        # 捕获前面代码抛出的异常，避免程序直接中断。
                        except Exception as e:
                            print(f"Could not parse trip check result: {e}")


# 定义函数 `display_trip_check_result`，把一段可复用逻辑封装起来。
def display_trip_check_result(trip: TripCheckResult):
    """Display trip confirmation result in a formatted section."""

    display(HTML(f"""
    <div style='padding: 20px; background: #f3e5f5; border-radius: 8px; margin: 15px 0; border-left: 4px solid #9c27b0;'>
        <h3 style='margin: 0 0 15px 0; color: #7b1fa2;'>🎯 Trip Confirmed</h3>
        <div style='display: grid; grid-template-columns: 1fr 1fr; gap: 15px; margin-bottom: 15px;'>
            <div>
                <strong style='color: #333;'>Trip Reference:</strong> {trip.trip_reference}<br>
                <strong style='color: #333;'>Destination:</strong> {trip.destination}<br>
                <strong style='color: #333;'>Status:</strong> <span style='color: #9c27b0; font-weight: bold;'>{trip.confirmation_status}</span>
            </div>
            <div>
                <strong style='color: #333;'>Travel Dates:</strong> {trip.travel_dates}<br>
                <strong style='color: #333;'>Contact Info:</strong> {trip.contact_info}
            </div>
        </div>
        <div style='margin-bottom: 10px;'>
            <strong style='color: #333;'>Special Notes:</strong> {trip.special_notes}
        </div>
        <div style='background: rgba(156,39,176,0.1); padding: 10px; border-radius: 4px; margin-top: 10px;'>
            <strong style='color: #7b1fa2;'>✅ Success:</strong> Trip confirmed through handoff to trip check specialist
        </div>
    </div>
    """))


# Run the trip check test
# 等待异步操作完成，再继续执行后续代码。
await test_trip_check_handoff()

   


[User]: Can you confirm my travel plans are all set?
[Workflow Status] IDLE


## 第九步：工作流程分析 - 理解交接流程


In [28]:
# 定义异步函数 `analyze_handoff_patterns`，用于处理需要 `await` 的流程。
async def analyze_handoff_patterns():
    """Analyze different handoff patterns and routing decisions."""

    display(HTML("""
    <div style='padding: 20px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #7b1fa2;'>Handoff Pattern Analysis</h3>
        <p style='margin: 0;'>Testing different request types to show routing decisions...</p>
    </div>
    """))

    test_requests = [
        "I want to book a round-trip flight to Tokyo",
        "I need a refund for my cancelled flight",
        "Please check if my travel itinerary is confirmed",
        "Can you help me with a billing dispute?"
    ]

    # 遍历这一组数据，逐条处理。
    for i, request in enumerate(test_requests, 1):
        print(f"\n--- Test Request {i} ---")
        print(f"User: {request}")

        # Run workflow and capture routing decision
        # 以流式方式启动工作流，边执行边接收事件。
        events = await drain_events(workflow.run(request, stream=True))

        # Analyze which agent was activated
        for event in events:
            # 根据当前条件决定后续走哪条逻辑分支。
            if event.type == "output":
                conversation = extract_messages(event.data)
                # 遍历这一组数据，逐条处理。
                for message in conversation:
                    # 根据当前条件决定后续走哪条逻辑分支。
                    if message.author_name == "customer_support_agent":
                        print(f"Support Agent: {message.text[:100]}...")
                    # 如果前面的条件不成立，再判断这个分支条件。
                    elif message.author_name in ["booking_agent", "disputes_agent", "trip_check_agent"]:
                        agent_type = {
                            "booking_agent": "🛫 BOOKING SPECIALIST",
                            "disputes_agent": "💰 DISPUTES SPECIALIST",
                            "trip_check_agent": "🎯 TRIP CHECK SPECIALIST"
                        }[message.author_name]
                        print(f"Routed to: {agent_type}")
                        break
                break
    display(HTML("""
    <div style='padding: 25px; background: linear-gradient(135deg, #9c27b0 0%, #673ab7 100%); color: white; border-radius: 12px; 
                box-shadow: 0 4px 12px rgba(156,39,176,0.4); margin: 20px 0;'>
        <h2 style='margin: 0 0 20px 0;'>Handoff Analysis Results</h2>
        <div style='background: rgba(255,255,255,0.15); padding: 15px; border-radius: 8px;'>
            <h4 style='margin: 0 0 10px 0;'>Key Observations</h4>
            <ul style='margin: 0; padding-left: 20px; line-height: 1.6;'>
                <li><strong>Dynamic Routing:</strong> Customer support agent analyzes request intent</li>
                <li><strong>Context Preservation:</strong> Full conversation history maintained</li>
                <li><strong>Specialist Focus:</strong> Each agent handles their expertise area</li>
                <li><strong>Seamless Handoff:</strong> Users don't need to repeat information</li>
            </ul>
        </div>
    </div>
    """))

    # Run the analysis
# 等待异步操作完成，再继续执行后续代码。
await analyze_handoff_patterns()



--- Test Request 1 ---
User: I want to book a round-trip flight to Tokyo

--- Test Request 2 ---
User: I need a refund for my cancelled flight

--- Test Request 3 ---
User: Please check if my travel itinerary is confirmed

--- Test Request 4 ---
User: Can you help me with a billing dispute?



---

**免责声明**：  
本文档使用AI翻译服务[Co-op Translator](https://github.com/Azure/co-op-translator)进行翻译。尽管我们努力确保翻译的准确性，但请注意，自动翻译可能包含错误或不准确之处。原始语言的文档应被视为权威来源。对于重要信息，建议使用专业人工翻译。我们不对因使用此翻译而产生的任何误解或误读承担责任。
